# W04-A · A sentiment classifier

A shop gets hundreds of reviews a day. Which ones are unhappy customers? Reading the **feeling** of a text, positive or
negative, is called **sentiment analysis**.

Today you build a sentiment classifier from 13 short reviews, using nothing but counting and multiplying: **Naïve
Bayes**. Every word is a small clue; the classifier multiplies the clues, and the bigger score wins.

By the end of this notebook you will be able to:
1. work out the **prior**: how common positive reviews are
2. count the words in the positive and the negative reviews
3. turn the counts into word probabilities, with **add-one smoothing**
4. score a review for each label
5. decide whether a review is positive or negative, and how sure the classifier is
6. find the words that are the strongest clues

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/shripada/ads6703-nlp/blob/main/w04/W04-A-sentiment-classifier.ipynb) · On Colab, run the next cell first. On your laptop, you can skip it.

In [ ]:
# Only for Google Colab: fetch the course folder (with the data) and move into this week's folder.
# On your laptop this cell does nothing.
import os, subprocess, sys
if "google.colab" in sys.modules:
    if os.path.exists("/content/ADS6703"):
        subprocess.run(["git", "-C", "/content/ADS6703", "pull", "-q"], check=True)
    else:
        subprocess.run(["git", "clone", "-q", "https://github.com/shripada/ads6703-nlp.git", "/content/ADS6703"], check=True)
    os.chdir("/content/ADS6703/w04")

In [ ]:
# ▶ Setup: just run this cell. These 13 reviews are all the text our classifier will ever read.
from collections import Counter   # Counter: a dictionary that counts things (see W03-0)

# The training set: 13 (text, label) pairs, 8 positive, then 5 negative.
# "Training" means the classifier learns from these reviews, and only these, by counting their words.
train = [
    ("great filter works well", "pos"),
    ("easy to install and works great", "pos"),
    ("love it great price", "pos"),
    ("works perfectly and fits well", "pos"),
    ("good value easy fit", "pos"),
    ("quiet and fast love it", "pos"),
    ("great product good price", "pos"),
    ("good quality and easy to use", "pos"),
    ("stopped working after a week", "neg"),
    ("leaks and broke fast", "neg"),
    ("bad fit not worth the money", "neg"),
    ("cheap and noisy not good", "neg"),
    ("does not work bad quality", "neg"),
]
print(len(train), "reviews")   # len() of a list: how many items it holds

## 1 · The prior: how common are positive reviews?

Before reading a single word, what is the chance a review is positive? Just the share of positive reviews in the
training set. This is the **prior**, like the 10 wolves in 1000 photos.

### ✍️ Exercise 1: `prior_pos` and `prior_neg`

Count the positive reviews, and divide by the number of reviews. The negative prior is what is left.

In [ ]:
n_pos = 0                        # the tally of positive reviews starts at zero
for text, label in train:        # unpack each (text, label) pair
    ...   # ✍️ add 1 to n_pos if the label is "pos"
prior_pos = ...   # ✍️
prior_neg = ...   # ✍️ the rest
# f"..." is an f-string: {prior_pos:.2f} puts in the value of prior_pos, rounded to 2 decimal places.
print(f"P(pos) = {prior_pos:.2f}   P(neg) = {prior_neg:.2f}")
# abs(a - b) < 1e-12 means "a equals b, give or take a tiny rounding error".
assert n_pos == 8 and abs(prior_pos - 8 / 13) < 1e-12, "not quite: 8 positive reviews out of 13"
print("✅")

## 2 · Count the words in each pile

### ✍️ Exercise 2: `pos_counts` and `neg_counts`

Fill two Counters: the words of the positive reviews, and the words of the negative reviews.

In [ ]:
pos_counts = Counter()                    # the words of the positive reviews
neg_counts = Counter()                    # the words of the negative reviews
for text, label in train:
    ...   # ✍️ add the words of the text to the right Counter
# .most_common(6): the 6 words with the biggest counts, as (word, count) pairs.
print("positive:", pos_counts.most_common(6))
print("negative:", neg_counts.most_common(6))
# Check: "great" is in 4 positive reviews, and "not" in 3 negative ones.
assert pos_counts["great"] == 4 and neg_counts["not"] == 3, "not quite"
print("✅")

In [ ]:
vocab = set(pos_counts) | set(neg_counts)   # every word the training reviews use, each once
V = len(vocab)                              # V: how many different words. Smoothing needs it.
print("V =", V, "different words")
# sum(c.values()) adds up all the counts in a Counter: the number of words in that pile.
print(sum(pos_counts.values()), "words in positive reviews,", sum(neg_counts.values()), "in negative reviews")

> 📖 **Reading the output.** *great* tops the positive pile and *not* the negative one. *and* is common in both: it
> carries no feeling. The positive pile has more words, simply because there are more positive reviews.

## 3 · From counts to word probabilities

P(great | pos) is the share of the positive pile that is *great*: count ÷ all the words in the pile. But *noisy*
never appears in a positive review, so P(noisy | pos) would be 0, and one zero wipes out a whole review's score (the
zero problem from W3). The fix is the same: **add-one smoothing**.

$$P(\text{word} \mid \text{label}) = \frac{\text{count of the word in that label} + 1}{\text{all words in that label} + V}$$

### ✍️ Exercise 3: `p_word(word, counts)`

`counts` is `pos_counts` or `neg_counts`. Return the add-one probability of `word` in that pile.

In [ ]:
def p_word(word, counts):
    # counts[word]: how often the word appears in this pile (0 if never).
    # sum(counts.values()): all the words in the pile.   V: how many different words there are.
    return ...   # ✍️ (count of the word + 1) / (all words in the pile + V)
# round(x, 3) rounds x to 3 decimal places, so it is easier to read.
print("P(great | pos) =", round(p_word("great", pos_counts), 3), "  P(great | neg) =", round(p_word("great", neg_counts), 3))
print("P(noisy | pos) =", round(p_word("noisy", pos_counts), 3), "  P(noisy | neg) =", round(p_word("noisy", neg_counts), 3))
# Check: "great" appears 4 times among the 38 positive words, and V is 37.
assert abs(p_word("great", pos_counts) - 5 / 75) < 1e-12, "not quite: (4 + 1) / (38 + 37)"

# The probabilities of all the words in one pile must still add up to 1
total = 0
for w in vocab:                               # every word we know
    total = total + p_word(w, pos_counts)     # add its probability in the positive pile
assert abs(total - 1) < 1e-9, "the probabilities in the positive pile should add up to 1"
print("✅")

> 📖 **Reading the output.** *great* is about 4 times as likely in a positive review as in a negative one; *noisy* is
> more than twice as likely in a negative one. And thanks to smoothing, no word gets exactly 0, not even *noisy* in
> the positive pile, where it never appears.

## 4 · Score a review

The score for a label is the prior times the probability of every word in the review:

**score(pos)** = P(pos) × P(word₁ | pos) × P(word₂ | pos) × …

Words the classifier never saw in training (not in `vocab`) tell us nothing either way, so we **skip** them.

### ✍️ Exercise 4: `score(review, prior, counts)`

In [ ]:
def score(review, prior, counts):
    s = prior                        # start with the prior: P(pos) or P(neg)
    for word in review.split():      # one round per word of the review
        if word not in vocab:
            continue                 # never seen in training: skip it
        ...   # ✍️ multiply s by this word's probability
    return s
# Score one review twice: with the positive prior and pile, then with the negative ones.
review = "great price but noisy"
print("positive score:", score(review, prior_pos, pos_counts))
print("negative score:", score(review, prior_neg, neg_counts))
# Check 1: a one-word review scores the prior × that word's probability.
assert abs(score("great", prior_pos, pos_counts) - 8 / 13 * 5 / 75) < 1e-12, "not quite"
# Check 2: "zzz" is not in vocab, so it is skipped and the score stays at the prior.
assert score("zzz", prior_pos, pos_counts) == prior_pos, "unknown words should be skipped"
print("✅")

> 📖 **Reading the output.** The scores are tiny, because they are products of small numbers. That doesn't matter:
> only **which one is bigger** does. (With long reviews the numbers get so tiny that real programs add logarithms
> instead of multiplying. We don't need that for short reviews.)

## 5 · Positive or negative?

### ✍️ Exercise 5: `classify(review)`

Return two things: the label with the bigger score (`"pos"` or `"neg"`), and the chance that the review is positive,
`pos_score / (pos_score + neg_score)`.

In [ ]:
def classify(review):
    pos_score = score(review, prior_pos, pos_counts)   # how well the review fits the positive pile
    neg_score = score(review, prior_neg, neg_counts)   # how well it fits the negative pile
    label = ...   # ✍️ "pos" if pos_score is bigger, otherwise "neg"
    return label, ...   # ✍️ and the chance of positive
# classify(...)[0] is the first of the two values it returns: the label.
assert classify("love it")[0] == "pos" and classify("broke and leaks")[0] == "neg", "not quite"
print("✅")
for r in ["love it", "broke and leaks", "great price but noisy", "easy to install but stopped working", "the"]:
    label, p = classify(r)                     # unpack the two values classify returns
    print(f"{label}  {p:.0%} positive   {r}")  # :.0% shows a share as a percentage: 0.73 becomes 73%

> 📖 **Reading the output.**
> - *love it*: positive, and very sure.
> - *broke and leaks*: negative, but only about 73% sure. Positive reviews are more common (the prior), and *and*
>   turns up more often in positive reviews, so both pull the other way.
> - *great price but noisy*: *great* and *price* outweigh *noisy*. Positive.
> - *easy to install but stopped working*: a **mistake**. *easy*, *to* and *install* all appear in positive reviews,
>   and outnumber *stopped* and *working*. The classifier counts clues; it doesn't know that *but* changes the story.
> - *the* appears once, in a negative review, so on its own it leans negative. A word with no feeling at all still
>   nudges the score, just from the reviews the classifier happened to read.

## 6 · Which words are the strongest clues?

For each word, divide P(word | pos) by P(word | neg). Far above 1: a positive clue. Far below 1: a negative clue.
Just run the cell.

In [ ]:
# ▶ Just run this cell.
# For every word: how many times more likely it is in a positive review than in a negative one.
ratio = {}                                   # an empty dictionary: word -> ratio
for w in vocab:
    ratio[w] = p_word(w, pos_counts) / p_word(w, neg_counts)

# key=ratio.get sorts the words by their ratio; reverse=True puts the biggest first.
ranked = sorted(vocab, key=ratio.get, reverse=True)   # biggest ratio first
# ranked[:5] is the first 5 words (the most positive); ranked[-5:] the last 5 (the most negative).
# [(w, round(ratio[w], 1)) for w in ...] builds a list with one (word, ratio) pair per word.
print("Most positive:", [(w, round(ratio[w], 1)) for w in ranked[:5]])
print("Most negative:", [(w, round(ratio[w], 2)) for w in ranked[-5:]])

> 📖 **Reading the output.** *great*, *works* and *easy* are the strongest positive clues; *not* and *bad* the strongest
> negative ones. That makes sense. But *a* is a negative clue too, as strong as *broke*: it appears once, in *"stopped
> working after a week"*. The classifier has no idea what words *mean*. It only knows which pile they turned up in, in
> 13 reviews.

## Recap

| idea | what you saw |
|---|---|
| Prior | 8 of 13 reviews are positive: P(pos) ≈ 0.62 |
| Word counts per label | *great* in the positive pile, *not* in the negative pile |
| P(word \| label) | (count + 1) ÷ (words in that label + V): add-one, so never 0 |
| Score | prior × every known word's probability; unknown words are skipped |
| Classify | the bigger score wins; pos ÷ (pos + neg) is how sure it is |
| Clue strength | P(word \| pos) ÷ P(word \| neg) |

In W04-B you test the classifier on 10 reviews it has never seen.